In [2]:
import pandas as pd
import numpy as np
import psycopg2
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


DB_HOST = "seal-users.c180so2u4aci.us-east-2.rds.amazonaws.com"
DB_NAME = "OLAP_Operations"
DB_USER = "postgres"
DB_PASS = "Nomelase123+"

def resolver_ceguera_marketing():
    print(" Conectando al cubo OLAP para extraer el histórico de usuarios...")
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASS,
            port=5432
        )
    except Exception as e:
        print(f" Error de conexión: {e}")
        return


    query = """
        SELECT 
            u.id_usuario,
            ub.pais,
            d.sistema_operativo,
            f_reg.dia_semana AS dia_registro,
            CASE 
                WHEN COUNT(CASE WHEN f_int.fecha > f_reg.fecha THEN 1 END) > 0 THEN 1
                ELSE 0
            END AS retencion_24h
        FROM hechos_adquisicion a
        JOIN dim_usuario u ON a.id_usuario = u.id_usuario
        JOIN dim_ubicacion ub ON a.id_ubicacion = ub.id_ubicacion
        JOIN dim_dispositivo d ON a.id_dispositivo = d.id_dispositivo
        JOIN dim_fecha_hora f_reg ON a.id_fecha_hora = f_reg.id_fecha_hora
        LEFT JOIN hechos_interacciones i ON u.id_usuario = i.id_usuario
        LEFT JOIN dim_fecha_hora f_int ON i.id_fecha_hora = f_int.id_fecha_hora
        GROUP BY u.id_usuario, ub.pais, d.sistema_operativo, f_reg.dia_semana, f_reg.fecha;
    """
    
    try:
        df = pd.read_sql_query(query, conn)
        conn.close()
    except Exception as e:
        print(f" Error al ejecutar la consulta: {e}")
        if 'conn' in locals(): conn.close()
        return

    if df.empty:
        print(" No hay datos suficientes para entrenar el modelo.")
        return

    print(f" Datos extraídos: {len(df)} usuarios para el análisis.")

    X = df[['pais', 'sistema_operativo', 'dia_registro']]
    y = df['retencion_24h']


    clases, conteos = np.unique(y, return_counts=True)
    if len(clases) < 2:
        print(" Error: Se necesita una muestra con usuarios retenidos (1) y no retenidos (0) para entrenar la regresión.")
        return


    categorical_features = ['pais', 'sistema_operativo', 'dia_registro']
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, categorical_features)
        ]
    )


    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])


    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(" Entrenando modelo de Regresión Logística...")
    model_pipeline.fit(X_train, y_train)


    feature_names = model_pipeline.named_steps['preprocessor'].transformers_[0][1].get_feature_names_out(categorical_features)
    coefficients = model_pipeline.named_steps['classifier'].coef_[0]
    
    df_pesos = pd.DataFrame({
        'Atributo': feature_names,
        'Impacto (Coeficiente)': coefficients
    }).sort_values(by='Impacto (Coeficiente)', ascending=False)

    print("\n" + "="*60)
    print("   IMPACTO DE VARIABLES EN LA RETENCIÓN (COEFICIENTES)")
    print("="*60)
    print(df_pesos.to_string(index=False))
    print("="*60)


    print("\n Calculando Score de Probabilidad de Retención para toda la base...")
    df['Score_Retencion'] = model_pipeline.predict_proba(X)[:, 1]


    df['Accion_Marketing'] = np.where(
        df['Score_Retencion'] < 0.3, 'SUSPENDER GASTO (Score < 0.3)',
        np.where(df['Score_Retencion'] > 0.7, 'INVERTIR PRESUPUESTO (Score > 0.7)', 'Monitorear / Neutro')
    )

    print("\n" + "="*60)
    print("   DISTRIBUCIÓN ESTRATÉGICA DEL PRESUPUESTO DE PUBLICIDAD")
    print("="*60)
    resumen = df['Accion_Marketing'].value_counts()
    for accion, total in resumen.items():
        porcentaje = (total / len(df)) * 100
        print(f"-> {accion}: {total} usuarios ({porcentaje:.2f}%)")
    print("="*60)

    output_file = "score_retencion_marketing.csv"
    df[['id_usuario', 'pais', 'sistema_operativo', 'Score_Retencion', 'Accion_Marketing']].to_csv(output_file, index=False)
    print(f"\n Archivo listo para sincronizar con Meta Ads / Google Ads: '{output_file}'\n")

if __name__ == "__main__":
    resolver_ceguera_marketing()

 Conectando al cubo OLAP para extraer el histórico de usuarios...


C:\Users\luigu\AppData\Local\Temp\ipykernel_28312\9526174.py:52: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


 Datos extraídos: 100000 usuarios para el análisis.
 Entrenando modelo de Regresión Logística...

   IMPACTO DE VARIABLES EN LA RETENCIÓN (COEFICIENTES)
                 Atributo  Impacto (Coeficiente)
         pais_New Zealand               4.631613
            pais_Colombia               4.602506
           pais_Indonesia               4.578629
           pais_Argentina               4.545304
              pais_France               4.519564
            pais_Ethiopia               4.477122
sistema_operativo_Android               0.414017
   dia_registro_Miércoles               0.414017
               pais_Egypt              -0.941186
        pais_South Africa              -0.950536
           pais_Australia              -0.951832
               pais_Italy              -0.958663
               pais_India              -0.963569
               pais_Japan              -0.978425
              pais_Canada              -0.986434
       pais_United States              -1.157046
      pais_Uni